# 04 — AutoAI Experiment + WML Deployment
**PIX Fraud Intelligence | IBM Portfolio Project**

**Goals:**
- Configure and run an AutoAI experiment via the `ibm_watsonx_ai` SDK
- Retrieve the pipeline leaderboard
- Deploy the best pipeline to a Watson Machine Learning endpoint
- Save deployment metadata for the evaluation notebook

> **Note:** AutoAI consumes CUH from your Watson Studio Lite quota (50 CUH/month).  
> A full experiment run uses ~5-10 CUH. Run sparingly.

> **WML Lite quota:** 20 CUH/month. The deployed endpoint persists without consuming  
> CUH — only predictions consume quota.

In [1]:
import sys
sys.path.append('..')

import json
import pandas as pd
from ibm_watsonx_ai import APIClient, Credentials
import warnings
warnings.filterwarnings('ignore')

## 1. Connect to Watson Machine Learning

In [2]:
with open('../config/credentials.json') as f:
    config = json.load(f)

wml_credentials = Credentials(
    api_key=config['wml']['apikey'],
    url=config['wml']['url']
)

client = APIClient(wml_credentials, space_id=config['wml']['space_id'])
print('WML client ready.')
print(f'ibm_watsonx_ai version: {client.version}')

WML client ready.
ibm_watsonx_ai version: 1.5.10


## 2. Upload Training Data to IBM Cloud Object Storage

In [3]:
# AutoAI reads training data from COS or inline
# We use the inline approach with the processed parquet
train_df = pd.read_parquet('../data/processed/transactions_features.parquet')
train_df = train_df[train_df['split'] == 'train'].drop(columns='split')
print(f'Training rows: {len(train_df):,} | Fraud rate: {train_df["fraude"].mean():.4%}')

# Save training CSV for upload
train_df.to_csv('../data/processed/autoai_train.csv', index=False)

# Upload to WML data assets
asset_details = client.data_assets.create(
    name='pix_fraud_train',
    file_path='../data/processed/autoai_train.csv'
)
training_data_asset_uid = client.data_assets.get_id(asset_details)
print(f'\nData asset uploaded. UID: {training_data_asset_uid}')

Training rows: 80,000 | Fraud rate: 13.7350%


Creating data asset...


SUCCESS

Data asset uploaded. UID: 019ecca2-8b8a-706f-959b-0d493a09062f


## 3. Configure AutoAI Experiment
AutoAI automatically:
- Selects best algorithms (XGBoost, LightGBM, Random Forest, etc.)
- Applies hyperparameter optimization
- Generates up to 4 pipeline variants per algorithm
- Ranks them by the chosen metric (F1 — best for imbalanced data)

In [4]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.helpers.connections import DataConnection

experiment = AutoAI(wml_credentials, space_id=config['wml']['space_id'])

pipeline_optimizer = experiment.optimizer(
    name='PIX Fraud Detection — AutoAI',
    prediction_type=AutoAI.PredictionType.BINARY,
    prediction_column='fraude',
    positive_label=1,
    scoring=AutoAI.Metrics.F1_SCORE,
    include_only_estimators=[
        AutoAI.ClassificationAlgorithms.XGB,
        AutoAI.ClassificationAlgorithms.LGBM,
        AutoAI.ClassificationAlgorithms.RF,
    ],
    max_number_of_estimators=3,
    train_sample_rows_test_size=0.1,
)

print('AutoAI optimizer configured:')
print(f'  Task:       Binary Classification')
print(f'  Target:     fraude (1 = fraud)')
print(f'  Metric:     F1 Score')
print(f'  Algorithms: XGBoost, LightGBM, Random Forest')

Note: Using `train_sample_rows_test_size` is deprecated.Use either `sample_rows_limit` or `sample_percentage_limit` instead.
Value of `train_sample_rows_test_size` parameter will be passed as `sample_percentage_limit`.


AutoAI optimizer configured:
  Task:       Binary Classification
  Target:     fraude (1 = fraud)
  Metric:     F1 Score
  Algorithms: XGBoost, LightGBM, Random Forest


## 4. Run AutoAI Experiment
> This cell takes 15-30 minutes. Monitor progress in Watson Studio UI.

In [5]:
# Reuse a previously completed AutoAI run with the same name if one exists
# (AutoAI consumes limited Lite-tier CUH — avoid re-spending on re-runs).
EXPERIMENT_NAME = 'PIX Fraud Detection — AutoAI'

runs_df = experiment.runs.list()
name_col = next((c for c in runs_df.columns if 'name' in c.lower()), None)
prior = runs_df[(runs_df[name_col] == EXPERIMENT_NAME) & (runs_df['state'] == 'completed')]

if len(prior):
    run_id = prior.iloc[0]['run_id']
    print(f'Reusing completed AutoAI run: {run_id}')
    pipeline_optimizer = experiment.runs.get_optimizer(run_id=run_id)
    run_details = pipeline_optimizer.get_run_details()
else:
    run_details = pipeline_optimizer.fit(
        training_data_reference=[
            DataConnection(data_asset_id=training_data_asset_uid)
        ],
        background_mode=False  # blocks until done
    )
    print('\nAutoAI experiment complete!')

print(f'Run ID: {run_details["metadata"]["id"]}')

Reusing completed AutoAI run: 048e9da2-cf9e-49eb-aed0-c9de1e2c89ca


Run ID: 048e9da2-cf9e-49eb-aed0-c9de1e2c89ca


## 5. Inspect Pipeline Leaderboard

In [6]:
summary = pipeline_optimizer.summary()

# Rank by holdout F1 (generalization), tie-break on the optimized training F1.
rank_cols = [c for c in ['holdout_f1', 'training_f1_(optimized)'] if c in summary.columns]
summary_sorted = summary.sort_values(rank_cols, ascending=False)

display_cols = [c for c in
                ['Enhancements', 'Estimator', 'holdout_f1', 'holdout_roc_auc',
                 'holdout_average_precision', 'training_f1_(optimized)']
                if c in summary.columns]
print('Pipeline Leaderboard (top 5 by holdout F1):')
print(summary_sorted[display_cols].head().to_string())

Pipeline Leaderboard (top 5 by holdout F1):
                         Enhancements                                      Estimator  holdout_f1  holdout_roc_auc  holdout_average_precision  training_f1_(optimized)
Pipeline Name                                                                                                                                                        
Pipeline_9               HPO, FE, HPO                                 LGBMClassifier         1.0         0.999104                        1.0                 0.937361
Pipeline_10    HPO, FE, HPO, Ensemble  BatchedTreeEnsembleClassifier(LGBMClassifier)         1.0         0.999104                        1.0                 0.937361
Pipeline_8                    HPO, FE                                 LGBMClassifier         1.0         0.999091                        1.0                 0.937071
Pipeline_7                        HPO                                 LGBMClassifier         1.0         0.999078             

## 6. Deploy Best Pipeline to WML Endpoint

In [7]:
from ibm_watsonx_ai.deployment import WebService

best_pipeline_name = summary_sorted.index[0]
print(f'Best pipeline: {best_pipeline_name} ({summary_sorted.loc[best_pipeline_name, "Estimator"]})')

DEPLOYMENT_NAME = 'PIX Fraud Detector — Online Endpoint'

# WebService stores the AutoAI pipeline and creates an online deployment in one step,
# resolving the correct software specification automatically (no hardcoded runtime).
service = WebService(
    source_instance_credentials=wml_credentials,
    source_space_id=config['wml']['space_id'],
    target_space_id=config['wml']['space_id'],
)
service.create(
    model=best_pipeline_name,
    deployment_name=DEPLOYMENT_NAME,
    experiment_run_id=run_details['metadata']['id'],
)

# Retrieve deployment + model identifiers and the scoring endpoint from the client
# (most robust across SDK versions).
matches = [d for d in client.deployments.get_details()['resources']
           if d['metadata']['name'] == DEPLOYMENT_NAME]
dep = sorted(matches, key=lambda d: d['metadata']['created_at'])[-1]
deployment_uid = dep['metadata']['id']
model_uid = dep['entity']['asset']['id']
scoring_url = client.deployments.get_scoring_href(dep)

print(f'\nModel UID:      {model_uid}')
print(f'Deployment UID: {deployment_uid}')
print(f'Scoring URL:    {scoring_url}')

Best pipeline: Pipeline_9 (LGBMClassifier)


Preparing an AutoAI Deployment...


Published model uid: 019ecca2-ed59-777f-80c0-f1767450b974
Deploying model 019ecca2-ed59-777f-80c0-f1767450b974 using V4 client.




######################################################################################

Synchronous deployment creation for id: '019ecca2-ed59-777f-80c0-f1767450b974' started

######################################################################################


initializing


Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.

...

.

.

.


ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='019ecca3-1019-7712-b599-995f8b32523f'
-----------------------------------------------------------------------------------------------





Model UID:      019ecca2-ed59-777f-80c0-f1767450b974
Deployment UID: 019ecca3-1019-7712-b599-995f8b32523f
Scoring URL:    https://us-south.ml.cloud.ibm.com/ml/v4/deployments/019ecca3-1019-7712-b599-995f8b32523f/predictions


## 7. Save Deployment Metadata

In [8]:
# Persist for notebook 05
deployment_meta = {
    'model_uid': model_uid,
    'deployment_uid': deployment_uid,
    'scoring_url': scoring_url,
    'best_pipeline': best_pipeline_name,
}

with open('../config/deployment_meta.json', 'w') as f:
    json.dump(deployment_meta, f, indent=2)

print('Deployment metadata saved to config/deployment_meta.json')
print(json.dumps(deployment_meta, indent=2))

Deployment metadata saved to config/deployment_meta.json
{
  "model_uid": "019ecca2-ed59-777f-80c0-f1767450b974",
  "deployment_uid": "019ecca3-1019-7712-b599-995f8b32523f",
  "scoring_url": "https://us-south.ml.cloud.ibm.com/ml/v4/deployments/019ecca3-1019-7712-b599-995f8b32523f/predictions",
  "best_pipeline": "Pipeline_9"
}


## 8. Quick Smoke Test
Score the first 3 test rows against the deployed endpoint.

In [9]:
test_df = pd.read_parquet('../data/processed/transactions_features.parquet')
test_df = test_df[test_df['split'] == 'test'].drop(columns=['split', 'fraude'])
sample = test_df.head(3)

payload = {'input_data': [{'values': sample.values.tolist()}]}
result = client.deployments.score(deployment_uid, payload)

print('Smoke test predictions:')
for i, pred in enumerate(result['predictions'][0]['values']):
    print(f'  Row {i}: prediction={pred[0]}, probability={pred[1]}')

Smoke test predictions:
  Row 0: prediction=0, probability=[0.9999895853472157, 1.041465278430783e-05]
  Row 1: prediction=0, probability=[0.9999888094588402, 1.1190541159852316e-05]
  Row 2: prediction=0, probability=[0.9999395436515047, 6.045634849531583e-05]
